In [ ]:
# ==========================================
#  TIME FREEDOM SCANNER + STOOQ FALLBACK
#  Daily Data: Yahoo → Stooq | Hourly: Twelve Data
# ==========================================

!pip install pandas requests yfinance pandas_datareader --quiet

import pandas as pd
import requests
import yfinance as yf
import warnings
import time
import numpy as np
from datetime import datetime, timedelta
import pandas_datareader.data as web

warnings.simplefilter(action='ignore', category=FutureWarning)

# ==========================================
# 1. CONFIGURATION
# ==========================================
TD_API_KEY = "7c5a55559d164d4ebca0d23a5baea839"   # ← Change if needed

LOOKBACK_DAYS   = 60      
DISCOUNT_LEVEL  = 0.20   
PREMIUM_LEVEL   = 0.80    
TARGET_PROCESSED = 80    
EARNINGS_BUFFER = 21     

MIN_PRICE       = 25     
MAX_PRICE       = 500    

TRIGGER_EMA = 8      
RSI_PERIOD  = 14     
RSI_BUY_MAX = 45     
RSI_SELL_MIN = 55    

# ==========================================
# TICKER LIST
# ==========================================
RAW_TICKERS = [
    "A", "AA", "AAL", "AAP", "AAPL", "ABBV", "ABNB", "ABT", "ACN", "ADBE", "ADI", "ADM", "ADP", "ADSK", "AEP", "AES", "AFL", "AFRM", "AGNC", "AI", "AIG", "AKAM", "ALB", "ALK", "ALL", "AMAT", "AMD", "AMGN", "AMRN", "AMZN", "APA", "APH", "APTV", "AVGO", "AXP", "BA", "BABA", "BAC", "BAX", "BBY", "BEN", "BIDU", "BIIB", "BITO", "BK", "BKR", "BMY", "BP", "BSX", "BX", "BYND", "C", "CAH", "CAT", "CB", "CCI", "CCJ", "CCL", "CF", "CFG", "CHWY", "CI", "CL", "CLF", "CLX", "CMCSA", "CME", "CNC", "CNP", "COF", "COIN", "COP", "COST", "CPB", "CPRI", "CRM", "CRON", "CRWD", "CSCO", "CSX", "CTVA", "CVNA", "CVS", "CVX", "CZR", "D", "DAL", "DD", "DE", "DELL", "DHI", "DHR", "DIA", "DIS", "DKNG", "DLR", "DOCU", "DOW", "DRI", "DVN", "DXC", "EA", "EBAY", "ED", "EEM", "EFA", "EIX", "EL", "EMR", "ENPH", "EOG", "EQR", "EQT", "ETSY", "EVRG", "EW", "EWJ", "EWW", "EWY", "EWZ", "EXC", "EXPE", "F", "FANG", "FAST", "FCX", "FDX", "FE", "FI", "FITB", "FOXA", "FSLR", "FTI", "FTV", "FXE", "FXI", "GD", "GDX", "GE", "GILD", "GLD", "GLW", "GM", "GME", "GOOG", "GOOGL", "GPRO", "GPS", "GS", "HAL", "HBAN", "HBI", "HCA", "HD", "HIG", "HLT", "HOG", "HOLX", "HON", "HPE", "HPQ", "HYG", "IBB", "IBIT", "IBM", "ICE", "IEF", "INTC", "IP", "IPG", "IRM", "IVZ", "IWM", "IYR", "JCI", "JD", "JETS", "JNJ", "JNPR", "JPM", "K", "KHC", "KMI", "KO", "KR", "KRE", "KSS", "KWEB", "LEN", "LI", "LKQ", "LLY", "LNC", "LOW", "LQD", "LUMN", "LUV", "LVS", "LYB", "LYFT", "MA", "MAR", "MARA", "MAS", "MCD", "MCHP", "MDLZ", "MDT", "MET", "META", "MGM", "MMM", "MO", "MOS", "MPC", "MRK", "MRNA", "MRVL", "MS", "MSFT", "MSTR", "MTB", "MTCH", "MU", "NCLH", "NEE", "NEM", "NET", "NFLX", "NKE", "NOW", "NRG", "NTAP", "NTES", "NVAX", "NVDA", "NWL", "NWSA", "O", "OIH", "OKE", "OMC", "ON", "ORCL", "OXY", "PARA", "PBR", "PDD", "PENN", "PEP", "PFE", "PFG", "PG", "PGR", "PINS", "PLTR", "PNC", "PPL", "PRU", "PSX", "PTON", "PYPL", "QCOM", "QQQ", "RBLX", "RCL", "RF", "RIG", "RIOT", "RIVN", "ROKU", "ROST", "RTX", "RUN", "SBUX", "SCHD", "SCHW", "SEDG", "SHOP", "SIRI", "SLB", "SLV", "SMCI", "SMH", "SNAP", "SNOW", "SO", "SOFI", "SOXL", "SOXS", "SPG", "SPX", "SPXL", "SPXS", "SPY", "SQQQ", "STX", "SWKS", "SYF", "SYY", "T", "TAP", "TBT", "TCOM", "TDOC", "TFC", "TGT", "TJX", "TLT", "TMO", "TMUS", "TPR", "TQQQ", "TRIP", "TRV", "TSLA", "TSM", "TSN", "TT", "TTD", "TTWO", "TXN", "U", "UA", "UAA", "UAL", "UBER", "ULTA", "UNG", "UNH", "UNP", "UPS", "UPST", "URBN", "USB", "USO", "UVXY", "V", "VALE", "VFC", "VGK", "VTR", "VXX", "VZ", "WAB", "WBA", "WBD", "WDC", "WFC", "WM", "WMB", "WMT", "WU", "WY", "WYNN", "X", "XBI", "XEL", "XHB", "XLB", "XLC", "XLE", "XLF", "XLI", "XLK", "XLP", "XLRE", "XLU", "XLV", "XLY", "XOM", "XOP", "XRT", "XRX", "XSP", "YELP", "YINN", "ZION", "ZM"
]
tickers = list(set(RAW_TICKERS))

# YAHOO TICKER MAPPING
TICKER_MAP = {
    "SPX": "^GSPC", "XSP": "^XSP", "DJI": "^DJI", "IXIC": "^IXIC", 
    "BRK.B": "BRK-B", "BF.B": "BF-B"
}

# ==========================================
# 2. INDICATOR FUNCTIONS
# ==========================================
def calculate_rsi(series, period=14):
    delta = series.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
    rs = gain / loss
    rsi = 100 - (100 / (1 + rs))
    return rsi.fillna(50)

def calculate_ema(series, span):
    return series.ewm(span=span, adjust=False).mean()

def check_earnings_safe(ticker):
    try:
        stock = yf.Ticker(ticker)
        cal = stock.calendar
        if cal is not None and not cal.empty:
            if isinstance(cal, pd.DataFrame) and 'Earnings Date' in cal.index:
                next_date = cal.loc['Earnings Date'].iloc[0]
            else:
                next_date = cal.get(0, None) if isinstance(cal, dict) else None
            
            if next_date:
                next_date = pd.to_datetime(next_date).replace(tzinfo=None)
                days = (next_date - datetime.now().replace(tzinfo=None)).days
                if days < 0: return True, 90
                return (days >= EARNINGS_BUFFER), days
    except:
        pass
    return True, 999

# ==========================================
# 3. DATA DOWNLOAD FUNCTIONS (with Stooq fallback)
# ==========================================
def get_yahoo_daily(tickers, days=150):
    print(f"   [YAHOO] Downloading daily data for {len(tickers)} symbols...")
    start_date = datetime.now() - timedelta(days=days)
    mapping = {t.replace(".", "-"): t for t in tickers}
    yahoo_tickers = list(mapping.keys())
    
    try:
        df = yf.download(yahoo_tickers, start=start_date, progress=False, threads=True, timeout=30)
        if df.empty:
            return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()
        
        close = df['Close'].rename(columns=mapping)
        high = df['High'].rename(columns=mapping)
        low = df['Low'].rename(columns=mapping)
        return close.sort_index(), high.sort_index(), low.sort_index()
    except Exception as e:
        print(f"   [YAHOO] Failed: {e}")
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

def get_stooq_daily(tickers, days=150):
    print(f"   [STOOQ] Downloading daily data for {len(tickers)} symbols (fallback)...")
    start = datetime.now() - timedelta(days=days)
    batch_size = 60
    close_data, high_data, low_data = {}, {}, {}
    
    for i in range(0, len(tickers), batch_size):
        batch = tickers[i:i+batch_size]
        try:
            df = web.DataReader(batch, 'stooq', start=start)
            if not df.empty:
                close_data.update(df['Close'].to_dict('series') if isinstance(df['Close'], pd.DataFrame) else {batch[0]: df['Close']})
                high_data.update(df['High'].to_dict('series') if isinstance(df['High'], pd.DataFrame) else {batch[0]: df['High']})
                low_data.update(df['Low'].to_dict('series') if isinstance(df['Low'], pd.DataFrame) else {batch[0]: df['Low']})
            time.sleep(0.5)
        except:
            pass
    
    close = pd.DataFrame(close_data)
    high = pd.DataFrame(high_data)
    low = pd.DataFrame(low_data)
    
    print(f"   [STOOQ] Success - Got data for {len(close.columns)} tickers")
    return close.sort_index(), high.sort_index(), low.sort_index()

# ==========================================
# 4. TWELVE DATA HOURLY
# ==========================================
def get_twelve_data_hourly(symbol):
    base_url = "https://api.twelvedata.com/time_series"
    params = {
        "symbol": symbol,
        "interval": "1h",
        "outputsize": 350,
        "apikey": TD_API_KEY,
        "order": "ASC"
    }
    try:
        resp = requests.get(base_url, params=params, timeout=15)
        data = resp.json()
        if "values" in data:
            df = pd.DataFrame(data["values"])
            df['datetime'] = pd.to_datetime(df['datetime'])
            df['Close'] = pd.to_numeric(df['close'])
            df.set_index('datetime', inplace=True)
            return df[['Close']]
        return pd.DataFrame()
    except:
        return pd.DataFrame()

# ==========================================
# 5. MAIN SCANNER
# ==========================================
def scan_market(ticker_list):
    print("\n📡 PHASE 1: Downloading Daily Data...")
    
    # Try Yahoo first
    df_close, df_high, df_low = get_yahoo_daily(ticker_list)
    
    # Fallback to Stooq if Yahoo fails
    if df_close.empty or len(df_close) < 30:
        print("⚠️  Yahoo failed → Switching to Stooq...")
        df_close, df_high, df_low = get_stooq_daily(ticker_list)
    
    if df_close.empty or len(df_close) < 30:
        print("❌ Both Yahoo and Stooq failed. Please try again later.")
        return pd.DataFrame()
    
    print(f"✅ Loaded daily data for {len(df_close.columns)} tickers.\n")
    
    # PHASE 1: Find extreme stocks
    all_metrics = []
    filtered_out_count = 0
    print("   [PHASE 1] Calculating price zones...")

    for ticker in df_close.columns:
        try:
            closes = df_close[ticker].dropna()
            if len(closes) < LOOKBACK_DAYS: 
                continue

            curr_close = float(closes.iloc[-1])
            if curr_close < MIN_PRICE or curr_close > MAX_PRICE:
                filtered_out_count += 1
                continue

            highs = df_high[ticker].dropna().tail(LOOKBACK_DAYS)
            lows = df_low[ticker].dropna().tail(LOOKBACK_DAYS)

            period_high = float(highs.max())
            period_low = float(lows.min())
            if period_high == period_low: 
                continue

            location = (curr_close - period_low) / (period_high - period_low)
            deviation = abs(location - 0.5)

            sig_type = 'BUY' if location <= DISCOUNT_LEVEL else 'SELL' if location >= PREMIUM_LEVEL else 'None'

            if sig_type != 'None':
                all_metrics.append({
                    'ticker': ticker,
                    'location': location,
                    'deviation': deviation,
                    'strike': period_low if location < 0.5 else period_high,
                    'price': curr_close,
                    'type': sig_type
                })
        except:
            continue

    all_metrics.sort(key=lambda x: x['deviation'], reverse=True)

    print(f"   -> Filtered out {filtered_out_count} stocks outside ${MIN_PRICE}-${MAX_PRICE}.")
    print(f"   -> Found {len(all_metrics)} candidates in extreme zones.\n")

    # PHASE 2: Process top candidates with earnings + 4H signals
    opportunities = []
    processed_count = 0
    skipped_count = 0

    print(f"   [PHASE 2] Checking Earnings & 4H signals for top {TARGET_PROCESSED}...")

    for cand in all_metrics:
        if processed_count >= TARGET_PROCESSED:
            break

        ticker = cand['ticker']
        is_safe, days = check_earnings_safe(ticker)

        if not is_safe:
            print(f"      [SKIP] {ticker} — Earnings in {days} days")
            skipped_count += 1
            continue

        processed_count += 1
        print(f"      {processed_count:2d}/{TARGET_PROCESSED}: {ticker} ({cand['type']}) ... ", end="")

        df_hourly = get_twelve_data_hourly(ticker)
        if df_hourly.empty:
            print("No hourly data")
            time.sleep(8)
            continue

        df_4h = df_hourly.resample('4h').last().dropna()
        if len(df_4h) < 20:
            print("Too few candles")
            time.sleep(8)
            continue

        df_4h['EMA8'] = calculate_ema(df_4h['Close'], TRIGGER_EMA)
        df_4h['RSI'] = calculate_rsi(df_4h['Close'], RSI_PERIOD)

        curr = df_4h['Close'].iloc[-1]
        ema = df_4h['EMA8'].iloc[-1]
        rsi = df_4h['RSI'].iloc[-1]

        sig = "None"
        status = ""

        if cand['type'] == 'BUY':
            if curr > ema and rsi < RSI_BUY_MAX:
                sig = "BUY"
                status = "CONFIRMED"
            else:
                status = f"Waiting (RSI {round(rsi,1)})"
        else:  # SELL
            if curr < ema and rsi > RSI_SELL_MIN:
                sig = "SELL"
                status = "CONFIRMED"
            else:
                status = f"Waiting (RSI {round(rsi,1)})"

        if sig != "None":
            print(f"+ {sig} SIGNAL!")
            opportunities.append({
                'Ticker': ticker,
                'Signal': sig,
                'Price': round(curr, 2),
                'Rec_Strike': round(cand['strike'], 2),
                'RSI': round(rsi, 1),
                'Link': f"https://finance.yahoo.com/quote/{ticker}"
            })
        else:
            print(f"- {status}")

        time.sleep(8)   # Twelve Data rate limit

    print(f"\n✅ Processed {processed_count} stocks | Skipped {skipped_count} due to earnings.")

    return pd.DataFrame(opportunities)

# ==========================================
# 6. RUN SCANNER
# ==========================================
if __name__ == "__main__":
    print("🚀 TIME FREEDOM SCANNER STARTED")
    print("="*80)
    
    results = scan_market(tickers)

    print("\n" + "="*80)
    print(f"TIME FREEDOM SCANNER — {datetime.now().strftime('%Y-%m-%d %H:%M')} ")
    print("="*80)

    if not results.empty:
        results = results.sort_values(by=['Signal', 'RSI'])
        print(results[['Ticker', 'Signal', 'Price', 'RSI', 'Rec_Strike', 'Link']].to_string(index=False))
        
        buys = results[results['Signal'] == "BUY"]['Ticker'].tolist()
        sells = results[results['Signal'] == "SELL"]['Ticker'].tolist()
        
        print("\n" + "-"*80)
        if buys: print(f"BUY WATCHLIST: {','.join(buys)}")
        if sells: print(f"SELL WATCHLIST: {','.join(sells)}")
    else:
        print("No confirmed signals found today.")

    print("\n✅ Scanner finished.")